# 22 — Figures and tables for the conference paper

Design log
* Reads only the committed CSVs of notebook 21 (`results/conference/`); no model is trained.
* Writes the same-capture paper figures (study design, Figure 1; same-capture results, Figure 2) at 600 dpi
  PNG and vector PDF, IEEE page width, paper names on every axis and legend, plus Table I values and the
  same-capture numbers quoted in the text, to `results/conference/paper/`. The transfer figures and
  tables come from notebook 23.
* Compute: CPU, seconds. No model API calls.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass
BASE = os.environ.get('DRIFT_BASE', '/content/drive/MyDrive/drift-conference')
RES = f'{BASE}/results/conference'
OUT = f'{RES}/paper'
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.lines import Line2D

A = pd.read_csv(f'{RES}/conf_demoA.csv')
SA = pd.read_csv(f'{RES}/conf_demoA_summary.csv')
U = pd.read_csv(f'{RES}/conf_unseen_summary.csv')
B = pd.read_csv(f'{RES}/conf_demoB.csv')
SB = pd.read_csv(f'{RES}/conf_demoB_summary.csv')
T = pd.read_csv(f'{RES}/conf_demoB_types.csv')
DS = pd.read_csv(f'{RES}/conf_drift_summary.csv')
AUD = pd.read_csv(f'{RES}/conf_label_audit.csv')
PAIRS = pd.read_csv(f'{RES}/conf_label_pairs.csv')

MODELS = ['rf', 'lgbm', 'mlp']
MODEL_NAME = {'rf': 'Random Forest', 'lgbm': 'LightGBM', 'mlp': 'MLP'}
DAY_NAME = {'regime_1': 'Day 1', 'regime_2': 'Day 2'}
COL = {'rf': '#1f5f99', 'lgbm': '#d9792b', 'mlp': '#3c8d3c'}
plt.rcParams.update({'font.family': 'serif', 'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7,
                     'legend.fontsize': 6.5, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
                     'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42,
                     'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02, 'hatch.linewidth': 0.5})
def save(fig, name):
    fig.savefig(f'{OUT}/{name}.pdf'); fig.savefig(f'{OUT}/{name}.png', dpi=600); plt.close(fig)
    print('saved', name)

## Figure 1 — Study design

In [ ]:
# Flows per CSE-CIC-IDS2018 capture day (Stage 1 schedule of notebook 21), used to draw the timelines to scale.
DAYS = [('14 Feb', 102334, 'Brute force'), ('15 Feb', 63016, 'DoS'), ('16 Feb', 85770, 'DoS'),
        ('20 Feb', 284890, 'DDoS'), ('21 Feb', 72909, 'DDoS'), ('22 Feb', 13260, 'Web'), ('23 Feb', 13348, 'Web'),
        ('28 Feb', 64609, 'Infiltration'), ('1 Mar', 64920, 'Infiltration'), ('2 Mar', 110895, 'Bot')]
assert sum(d[1] for d in DAYS) == 875951
GCOL = {'Brute force': '#8c6bb1', 'DoS': '#e6550d', 'DDoS': '#fd8d3c', 'Web': '#31a354',
        'Infiltration': '#6baed6', 'Bot': '#3182bd'}
C_TR, C_UN, C_TE, C_BUF = '#4c78a8', '#d9d9d9', '#f58518', '#54a24b'

W, H = 7.16, 2.3
fig = plt.figure(figsize=(W, H))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 38); ax.axis('off')
def fx(x): return x / 100
def fy(y): return y / 38
def band(x0, x1, title, c):
    ax.add_patch(FancyBboxPatch((x0, 0.6), x1 - x0, 36.8, boxstyle='round,pad=0.15,rounding_size=1.0',
                                fc=c, ec='#9aa5b1', lw=0.6))
    ax.text((x0 + x1) / 2, 35.6, title, ha='center', va='center', fontsize=7.5, weight='bold')
def card(x, y, w, h, fc='white'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12,rounding_size=0.5', fc=fc, ec='#55606b', lw=0.55))
def bar(x, y, w, h, parts, hatch_first=False):
    # parts: list of (fraction, colour)
    cx = x
    for i, (f, c) in enumerate(parts):
        ax.add_patch(Rectangle((cx, y), w * f, h, fc=c, ec='white', lw=0.3,
                               hatch='/////' if (hatch_first and i == 0) else None))
        cx += w * f
    ax.add_patch(Rectangle((x, y), w, h, fc='none', ec='#55606b', lw=0.4))
def arrow(x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='-|>', lw=0.6, color='#44505c',
                                                                  shrinkA=0, shrinkB=0))

band(0.4, 25.0, '1  Data', '#e8eef6'); band(26.0, 55.0, '2  Evaluation settings', '#eef6ea')
band(56.0, 74.6, '3  Models', '#fbf0e4'); band(75.6, 99.6, '4  What is measured', '#f3ecf7')

# --- 1 Data ---
tot = sum(d[1] for d in DAYS)
card(1.6, 21.4, 22.2, 12.6)
ax.text(12.7, 32.7, 'CSE-CIC-IDS2018', ha='center', fontsize=6.3, weight='bold')
ax.text(12.7, 31.2, 'training network: 875,951 flows,\n15 classes, 68 features', ha='center', va='top', fontsize=5.8,
        linespacing=1.1)
bar(2.6, 24.6, 20.2, 2.0, [(d[1] / tot, GCOL[d[2]]) for d in DAYS])
ax.text(2.6, 24.1, '14 Feb', fontsize=5.5, va='top'); ax.text(22.8, 24.1, '2 Mar 2018', fontsize=5.5, va='top', ha='right')

card(1.6, 7.6, 22.2, 12.9)
ax.text(12.7, 19.2, 'CIC-DDoS2019 (new network)', ha='center', fontsize=6.3, weight='bold')
ax.text(12.7, 17.2, '45 shared features; benign vs DDoS', ha='center', fontsize=6)
bar(2.6, 12.9, 20.2, 2.0, [(0.10, C_BUF), (0.90, '#bfbfbf')], hatch_first=True)
ax.text(2.6, 15.3, 'Day 1: 2.10M flows, 12 attack types', fontsize=5.7)
bar(2.6, 8.6, 20.2 * 0.98 / 2.10, 2.0, [(0.10, C_BUF), (0.90, '#bfbfbf')], hatch_first=True)
ax.text(2.6, 11.0, 'Day 2: 0.98M flows, 6 attack types', fontsize=5.7)

card(1.6, 1.4, 22.2, 5.2, fc='#fff8e6')
ax.text(12.7, 5.0, 'Label audit', ha='center', fontsize=6.3, weight='bold')
ax.text(12.7, 2.6, 'identical flows carrying different labels', ha='center', fontsize=5.8)

# --- 2 Evaluation settings ---
rng = np.random.default_rng(3)
x0, w = 27.6, 25.8
def setting(y, title):
    ax.text(x0, y, title, fontsize=6.2, weight='bold', va='bottom')
setting(29.9, 'Random split (same capture)')
cols = rng.choice([C_TR, C_TR, C_TR, C_UN, C_TE], size=60)
bar(x0, 27.5, w, 2.0, [(1 / 60, c) for c in cols])
setting(23.9, 'Time split, all classes seen')
for k, yy in enumerate([22.2, 20.8, 19.4]):
    bar(x0 + 3.4, yy, w - 3.4, 1.0, [(0.6, C_TR), (0.2, C_UN), (0.2, C_TE)])
    ax.text(x0 + 3.0, yy + 0.5, ['class 1', 'class 2', 'class 3'][k], fontsize=5.3, ha='right', va='center')
setting(16.0, 'Time split, later attacks unseen')
bar(x0, 13.6, w, 2.0, [(0.6, C_TR), (0.2, C_UN), (0.2, C_TE)])
ax.text(x0 + w * 0.9, 12.8, 'Bot, Infiltration:\nnever in training', fontsize=5.4, ha='center', va='top')
setting(7.6, 'New network (2018 to 2019)')
bar(x0, 5.0, w * 0.22, 2.0, [(1.0, C_TR)])
ax.text(x0 + w * 0.11, 4.2, '2018\ntraining', fontsize=5.4, ha='center', va='top')
arrow(x0 + w * 0.24, 6.0, x0 + w * 0.31, 6.0)
bar(x0 + w * 0.33, 5.0, w * 0.67, 2.0, [(0.12, C_BUF), (0.88, C_TE)], hatch_first=True)
ax.text(x0 + w * 0.33, 4.2, 'first 1-10%\nlabelled', fontsize=5.4, va='top')
ax.text(x0 + w, 4.2, 'rest of day:\nevaluation', fontsize=5.4, va='top', ha='right')
lg = [Rectangle((0, 0), 1, 1, fc=C_TR), Rectangle((0, 0), 1, 1, fc=C_UN), Rectangle((0, 0), 1, 1, fc=C_TE),
      Rectangle((0, 0), 1, 1, fc=C_BUF, hatch='/////')]
leg = ax.legend(lg, ['train', 'unused', 'test', 'labelled'], ncol=4, loc='upper center', frameon=False,
                bbox_to_anchor=(fx(40.5), fy(34.6)), fontsize=5.6, handlelength=1.2, columnspacing=0.8, handletextpad=0.4)

# --- 3 Models ---
card(57.2, 23.4, 16.2, 10.6)
ax.text(65.3, 30.6, 'Random Forest\nLightGBM\nMLP', ha='center', va='center', fontsize=6.4, linespacing=1.25)
ax.text(65.3, 25.3, '3 seeds, fixed settings,\nno tuning per setting', ha='center', va='center', fontsize=5.6)
card(57.2, 1.6, 16.2, 20.4)
ax.text(65.3, 20.4, 'Recalibration', ha='center', fontsize=6.3, weight='bold')
ia = fig.add_axes([fx(59.6), fy(8.6), fx(12.4), fy(10.0)])
s = np.linspace(0, 1, 200)
ia.plot(s, 1 / (1 + np.exp(-12 * (s - 0.35))), color='#1f5f99', lw=0.9, label='Platt')
ia.step([0, .15, .35, .5, .7, 1.0], [0.05, 0.05, 0.4, 0.75, 0.92, 0.92], where='post', color='#d9792b', lw=0.9, ls='--', label='Isotonic')
ia.set_xticks([]); ia.set_yticks([]); ia.set_xlabel('detector score', fontsize=5.4, labelpad=1)
ia.set_ylabel('probability', fontsize=5.4, labelpad=1)
ia.legend(fontsize=5.0, frameon=False, loc='lower right', handlelength=1.4)
ax.text(65.3, 5.0, 'refitted on the labelled\nsample; the model is frozen,\nthe order of scores is kept', ha='center',
        va='center', fontsize=5.5, linespacing=1.15)

# --- 4 What is measured ---
def hist_ax(x, y, w_, h_, thr=False):
    a = fig.add_axes([fx(x), fy(y), fx(w_), fy(h_)])
    g = np.linspace(-4, 6, 300)
    a.fill_between(g, np.exp(-(g + 0.3) ** 2 / 2), color='#9e9e9e', alpha=0.6, lw=0)
    a.fill_between(g, 0.8 * np.exp(-(g - 2.3) ** 2 / 2), color='#d6604d', alpha=0.6, lw=0)
    if thr:
        a.axvline(1.1, color='black', lw=0.8, ls='--')
    a.set_xticks([]); a.set_yticks([]); a.spines['left'].set_visible(False); a.set_xlim(-4, 6)
    return a
rows = [(28.2, 'Ranking', 'AUROC, AUPRC;\nalso per attack type'),
        (19.9, 'Decisions', 'MCC, macro-F1,\nDR, FAR at 0.5'),
        (11.6, 'Calibration', 'ECE: do stated\nprobabilities match?'),
        (3.3, 'Drift', 'PSI, KS per feature,\nday and traffic type')]
for i, (y, t1, t2) in enumerate(rows):
    card(76.8, y - 1.5, 21.6, 7.3)
    ax.text(85.2, y + 4.1, t1, fontsize=6.3, weight='bold', va='center')
    ax.text(85.2, y + 1.2, t2, fontsize=5.5, va='center', linespacing=1.15)
    if i == 0:
        a = hist_ax(77.6, y - 0.8, 7.0, 5.6)
    elif i == 1:
        a = hist_ax(77.6, y - 0.8, 7.0, 5.6, thr=True)
    elif i == 2:
        a = fig.add_axes([fx(78.6), fy(y - 0.9), fx(5.0), fy(5.8)])
        a.plot([0, 1], [0, 1], color='#9e9e9e', lw=0.6, ls=':')
        a.step([0, .2, .4, .6, .8, 1], [0.05, 0.12, 0.45, 0.5, 0.88, 0.95], where='post', color='#7b3294', lw=0.9)
        a.set_xticks([]); a.set_yticks([])
    else:
        a = fig.add_axes([fx(77.6), fy(y - 0.8), fx(7.0), fy(5.6)])
        g = np.linspace(-4, 7, 300)
        a.plot(g, np.exp(-g ** 2 / 2), color='#4c78a8', lw=0.9)
        a.plot(g, np.exp(-(g - 2.6) ** 2 / 3), color='#f58518', lw=0.9)
        a.set_xticks([]); a.set_yticks([]); a.spines['left'].set_visible(False)

# arrows between bands
for (xa, ya, xb, yb) in [(23.8, 27.5, 27.4, 28.5), (23.8, 27.5, 27.3, 24.6), (23.8, 27.5, 27.4, 14.6),
                         (23.8, 12.0, 27.4, 6.0), (53.6, 21.5, 57.0, 28.0), (53.6, 6.0, 57.0, 12.0),
                         (73.5, 28.0, 76.6, 30.5), (73.5, 12.0, 76.6, 14.5)]:
    arrow(xa, ya, xb, yb)
save(fig, 'fig1_study_design')

## Figure 2 — Same capture: time order, unseen attacks and label errors

In [ ]:
SPLITS = ['random', 'time_all_seen', 'time_global']
XL = ['Random', 'Time,\nall seen', 'Time,\nunseen']
nat = SA[SA.variant == 'native']
fig, axes = plt.subplots(1, 3, figsize=(7.16, 1.5), gridspec_kw={'width_ratios': [3, 3, 2.1]})
w = 0.26; xs = np.arange(3)
for i, m in enumerate(MODELS):
    s = nat[nat.model == m].set_index('split').loc[SPLITS]
    axes[0].bar(xs + (i - 1) * w, s['macro_f1_mean'], w, yerr=s['macro_f1_std'], color=COL[m], capsize=1.2,
                error_kw={'lw': 0.5}, label=MODEL_NAME[m])
    axes[1].bar(xs + (i - 1) * w, s['attack_recall_mean'], w, yerr=s['attack_recall_std'], color=COL[m], capsize=1.2,
                error_kw={'lw': 0.5})
    axes[1].plot(xs + (i - 1) * w, s['bin_auroc_mean'], ls='none', marker='D', ms=3.0, mfc='white', mec='black',
                 mew=0.7, label='AUROC (ranking)' if i == 0 else None)
axes[0].set_title('(a) Macro-F1', loc='left')
axes[1].set_title('(b) Detection rate (bars) and AUROC', loc='left')
for a in axes[:2]:
    a.set_xticks(xs); a.set_xticklabels(XL); a.set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
mg = SA[SA.variant == 'merged'].set_index('model'); nt = nat[nat.split == 'random'].set_index('model')
xs2 = np.arange(2); w2 = 0.36
for j, m in enumerate(['rf', 'lgbm']):
    axes[2].bar(j - w2 / 2, nt.loc[m, 'macro_f1_mean'], w2, color=COL[m], alpha=0.45, edgecolor=COL[m], lw=0.6)
    axes[2].bar(j + w2 / 2, mg.loc[m, 'macro_f1_mean'], w2, color=COL[m], edgecolor=COL[m], lw=0.6)
    axes[2].plot([j - w2 / 2, j + w2 / 2], [nt.loc[m, 'mcc_mean'], mg.loc[m, 'mcc_mean']], ls='none', marker='o', ms=3,
                 mfc='black', mec='black')
axes[2].set_xticks(xs2); axes[2].set_xticklabels(['Random\nForest', 'LightGBM']); axes[2].set_ylim(0, 1.05)
axes[2].set_title('(c) Two mislabelled classes\nmerged (random split)', loc='left')
h0, l0 = axes[0].get_legend_handles_labels(); h1, l1 = axes[1].get_legend_handles_labels()
hm = [Line2D([], [], ls='none', marker='o', ms=3, mfc='black', mec='black'),
      Rectangle((0, 0), 1, 1, fc='#9e9e9e', alpha=0.45, ec='#9e9e9e'), Rectangle((0, 0), 1, 1, fc='#9e9e9e', ec='#9e9e9e')]
fig.legend(h0 + h1 + hm, l0 + l1 + ['MCC', '(c) labels as given', '(c) merged'], ncol=7, frameon=False,
           loc='upper center', bbox_to_anchor=(0.5, 1.10), columnspacing=1.0)
fig.tight_layout(w_pad=1.0)
save(fig, 'fig2_same_capture')

## Table I values and same-capture numbers quoted in the text

In [ ]:
t1 = SA[['variant', 'split', 'model', 'macro_f1_mean', 'macro_f1_std', 'mcc_mean', 'mcc_std', 'attack_recall_mean',
         'fp_rate_mean', 'bin_auroc_mean', 'ece_mean']]
t1.round(3).to_csv(f'{OUT}/table1_same_capture.csv', index=False)
print(t1.round(3).to_string(index=False)); print()
nums = {}
for _, r_ in AUD.iterrows():
    nums[f'audit_{r_.label}'] = f'{int(r_.n_shared)}/{int(r_.n)}'
nums['pair_benign_infilteration_rows'] = int(PAIRS[(PAIRS.label_a == 'Benign') & (PAIRS.label_b == 'Infilteration')].rows.iloc[0])
sd_other = SA[SA.variant == 'native'][['mcc_std', 'attack_recall_std', 'fp_rate_std', 'bin_auroc_std', 'ece_std']].max().max()
nums['table1_max_sd_other_columns'] = float(sd_other)
for lab in U.label.unique():
    g = A[(A.variant == 'native') & (A.split == 'time_global')]
    nums[f'unseen_{lab}_max_flagged_any_seed'] = float(pd.read_csv(f'{RES}/conf_demoA_classes.csv').query("variant == 'native' and split == 'time_global' and label == @lab").flagged_attack.max())
N = pd.Series(nums, name='value'); N.to_csv(f'{OUT}/paper_numbers_same_capture.csv')
print(N.to_string()); print(); print(U.round(4).to_string(index=False))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "22: same-capture paper figures and Table I values from notebook 21 results"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)